In [ ]:
UPDATE_INTERVAL = 0.5  # Update every 0.5 seconds

print("Starting evacuation response loop...")
print(f"Update interval: {UPDATE_INTERVAL}s\n")

loop_count = 0
last_update_time = time.time()

try:
    while True:
        loop_count += 1
        current_time = time.time()
        elapsed = current_time - last_update_time
        
        with people_lock:
            # Update positions for each person
            for person_id in people:
                person = people[person_id]
                
                if person["evacuating"]:
                    # Move towards Torv
                    person["evacuation_progress"] = min(
                        person["evacuation_progress"] + elapsed / EVACUATION_TIME,
                        1.0
                    )
                    if person["evacuation_progress"] >= 1.0:
                        person["location"] = "torv"
                        person["evacuation_progress"] = 0.0
                else:
                    # Move back towards Strand (if at Torv)
                    if person["location"] == "torv":
                        person["evacuation_progress"] = min(
                            person["evacuation_progress"] + elapsed / RETURN_TIME,
                            1.0
                        )
                        if person["evacuation_progress"] >= 1.0:
                            person["location"] = "strand"
                            person["evacuation_progress"] = 0.0
        
        if loop_count % 2 == 0:  # Print status every 1 second
            print(f"[{loop_count}] Position update")
            publish_status()
        
        last_update_time = current_time
        time.sleep(UPDATE_INTERVAL)

except KeyboardInterrupt:
    print("\n\nResponse loop stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

## Response Loop

Main loop that:
1. Processes evacuation/return commands
2. Updates pedestrian positions
3. Reports status to dashboard

In [ ]:
people_lock = threading.Lock()

def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def get_person_location_coords(person_id):
    """Return current coordinates of a person."""
    with people_lock:
        person = people[person_id]
        if person["location"] == "torv":
            return KOEGE_TORCH_LAT, KOEGE_TORCH_LON
        else:
            # Interpolate during evacuation
            progress = person["evacuation_progress"]
            if person["evacuating"]:
                # Moving from strand to torv
                lat = KOEGE_STRAND_LAT + (KOEGE_TORCH_LAT - KOEGE_STRAND_LAT) * progress
                lon = KOEGE_STRAND_LON + (KOEGE_TORCH_LON - KOEGE_STRAND_LON) * progress
            else:
                # Returning from torv to strand
                lat = KOEGE_TORCH_LAT + (KOEGE_STRAND_LAT - KOEGE_TORCH_LAT) * progress
                lon = KOEGE_TORCH_LON + (KOEGE_STRAND_LON - KOEGE_TORCH_LON) * progress
            return lat, lon

def publish_status():
    """Publish current evacuation status of all people."""
    with people_lock:
        evacuated = sum(1 for p in people.values() if p["location"] == "torv")
        in_transit = sum(1 for p in people.values() if p["evacuating"] or p["evacuation_progress"] > 0)
    
    status_msg = f"Evacuated: {evacuated}/{NUM_PEOPLE} | In transit: {in_transit}"
    print(f"  {status_msg}")

def on_control_command(client, userdata, msg):
    """Callback when control command received."""
    try:
        data = json.loads(msg.payload.decode())
        cmd = ControlCommand.from_json(data)
        
        print(f"[{cmd.timestamp}] Control command received: {cmd.action} ({cmd.parameters.get('severity', '?')})")
        
        if cmd.action == "alert":
            severity = cmd.parameters.get("severity", "low")
            if severity == "high":
                print("  ⚠️  Initiating EVACUATION")
                with people_lock:
                    for person_id in people:
                        people[person_id]["evacuating"] = True
            else:
                print("  ✓ Initiating RETURN to normal activity")
                with people_lock:
                    for person_id in people:
                        people[person_id]["evacuating"] = False
    except Exception as e:
        print(f"Error parsing control command: {e}")

# Subscribe to control commands
connector.client.on_message = on_control_command
control_topic = make_topic(mqtt_cfg, "control", "command")
connector.client.subscribe(control_topic)

print(f"Subscribed to: {control_topic}")

## Evacuation Logic and Callbacks

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="response")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

## Connect to MQTT Broker

In [ ]:
import time
import json
import threading
from datetime import datetime, timezone

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import ControlCommand, ResponseStatus

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt

# Locations
KOEGE_TORCH_LAT = 55.456553861769855
KOEGE_TORCH_LON = 12.181774944848945
KOEGE_STRAND_LAT = 55.45058843620187
KOEGE_STRAND_LON = 12.197503222443261

# Pedestrian configuration
NUM_PEOPLE = 10
EVACUATION_TIME = 8.0  # seconds to reach safe zone
RETURN_TIME = 5.0  # seconds to return

# Pedestrian state
people = {}
for i in range(NUM_PEOPLE):
    people[f"person_{i+1}"] = {
        "location": "strand",  # "strand" or "torv"
        "evacuating": False,
        "evacuation_progress": 0.0  # 0.0 to 1.0
    }

print(f"Response agent configured:")
print(f"  Køge Torv (safe): ({KOEGE_TORCH_LAT}, {KOEGE_TORCH_LON})")
print(f"  Køge Strand (danger): ({KOEGE_STRAND_LAT}, {KOEGE_STRAND_LON})")
print(f"  Pedestrians: {NUM_PEOPLE}")
print(f"  Evacuation time: {EVACUATION_TIME}s")
print(f"  Return time: {RETURN_TIME}s")

## Setup and Configuration

# Agent: Response (Evacuation & Pedprogram)

**Role:** Executes evacuation directives and manages pedestrian movement.

**Responsibilities:**
1. Listen for alert commands from Control agent
2. Track evacuation status of 10 people
3. Simulate movement from Køge Søndre Strand (beach) → Køge Torv (safe zone)
4. Resume normal activity when all-clear issued
5. Publish status updates to dashboard

**People Management:**
- 10 pedestrians initially at Køge Søndre Strand
- On HIGH alert: move to Køge Torv (takes ~8 seconds)
- On LOW alert: return to Køge Søndre Strand (takes ~5 seconds)
- Report location and evacuation status

# Agent Response
Actuator notebook that listens for `ControlCommand` messages and publishes `ResponseStatus` updates.